# Character Rewrite

This is Phase 2 of converting OCR'd source to `opend6_tools`.
This phase proceeds in three steps:

1. Load the TOML version of the text created by Phase 1: Character Extract.
2. Create some Python code that will create the Character or Creature, populated with appropriate details from the TOML.
3. Use `opend6_tools.notebook_extract.ModuleWriter` to emit a proper character module.

The first step will tend to uncover subtle parsing errors left over from phase 1 processing.
Fixing these, means touching the source file to fix the problem, or (possibly) editing the parser that implements phase 1.

The result isn't going to work. It's merely a starting point for further cleanup.

In [5]:
from pathlib import Path
from dataclasses import dataclass, field, asdict
import tomllib
from pprint import pprint

from opend6_tools.notebook_extract import ModuleWriter, write_characters, slug

import character_source as src

base_dir = Path.cwd().parent

def character_load_iter(path):
    for char in tomllib.loads(path.read_text())['characters']:
        try:
            character = src.Character.from_dict(char)
            # print(character.name)
            yield character
        except Exception:
            print(p.parent.parent.name, '/*/', p.name)
            print(char)
            raise

def od6_obj_iter(char_iter):
    for char in char_iter:
        try:
            od6_object = char.d6obj()
            yield od6_object.name, od6_object
        except Exception:
            print("##", char.name)
            pprint(char, width=256)
            print()
            raise

def pycode_iter(char_iter):
    for char in char_iter:
        try:
            name_slug, code_block = char.pycode()
        except Exception:
            print("##", char.name)
            pprint(char, width=256)
            print()
            raise
        yield name_slug, code_block

In [6]:
character_paths = base_dir.glob("*/characters/*.toml")
mw = ModuleWriter('character_rewrite notebook')
for p in character_paths:
    characters = list(pycode_iter(character_load_iter(p)))
    target = p.with_suffix(".py")
    print(target)
    write_characters('characters', target, p.stem, characters, mw)

/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/fantasy_locations/Settlements/characters/guild_member.py
/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/fantasy_locations/Settlements/characters/kasen.py
/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/fantasy_locations/Settlements/characters/library_guard.py
/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/fantasy_locations/Settlements/characters/phylo_duran.py
/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/fantasy_locations/Settlements/characters/mukhtar.py
/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/fantasy_locations/Settlements/characters/balam_mis.py
/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/fantasy_locations/Settlements/characters/philsopher.py
/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/fantasy_locations/Settlements/characters/ori_swifthand.py
/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/fantasy_locations/Settlements/characters/cire.py
/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/f